# Automatic Bubble Mask Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliRKhojasteh/Bubble_segmentation/blob/main/Notebooks/automatic_mask_generator.ipynb)

This notebook uses **SAM2** (Segment Anything Model 2) to automatically generate segmentation masks for bubbles in experimental flow images.

**Workflow:**
1. Install dependencies and download model checkpoints (automatic)
2. Configure the mask generator parameters
3. Run on a single demo image or batch-process a directory

> **Reference:** Khojasteh, A.R., van de Water, W. & Westerweel, J. (2024). *Practical object and flow structure segmentation using artificial intelligence.* Experiments in Fluids, 65, 119. [DOI: 10.1007/s00348-024-03852-7](https://doi.org/10.1007/s00348-024-03852-7)

## 1. Setup and Installation
Run this cell to install SAM2 and download model checkpoints. This is handled automatically on Google Colab.

In [ ]:
import os
import subprocess
import sys

# --- Detect environment ---
IN_COLAB = "google.colab" in sys.modules

# --- Resolve repository root (works from Notebooks/ or repo root) ---
if IN_COLAB:
    REPO_ROOT = "."
elif os.path.basename(os.getcwd()) == "Notebooks":
    REPO_ROOT = os.path.abspath("..")
else:
    REPO_ROOT = os.path.abspath(".")

# --- Install SAM2 if not available ---
try:
    import sam2
    print("SAM2 already installed.")
except ImportError:
    print("Installing SAM2...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 
                   "git+https://github.com/facebookresearch/sam2.git"], check=True)
    print("SAM2 installed successfully.")

# --- Download checkpoints if not present ---
CHECKPOINT_DIR = os.path.join(REPO_ROOT, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

checkpoints = {
    "sam2.1_hiera_tiny.pt":      "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt",
    "sam2.1_hiera_small.pt":     "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt",
    "sam2.1_hiera_base_plus.pt": "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt",
    "sam2.1_hiera_large.pt":     "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt",
}

# Only download the model you plan to use (default: large)
MODELS_TO_DOWNLOAD = ["sam2.1_hiera_large.pt"]  # Add more if needed

for ckpt_name in MODELS_TO_DOWNLOAD:
    ckpt_path = os.path.join(CHECKPOINT_DIR, ckpt_name)
    if not os.path.exists(ckpt_path):
        print(f"Downloading {ckpt_name}...")
        import urllib.request
        urllib.request.urlretrieve(checkpoints[ckpt_name], ckpt_path)
        print(f"  Saved to {ckpt_path}")
    else:
        print(f"Checkpoint found: {ckpt_path}")

# --- Download demo images if on Colab ---
DEMO_DIR = os.path.join(REPO_ROOT, "Demo")
if IN_COLAB and not os.path.exists(DEMO_DIR):
    print("Downloading demo images...")
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                   "https://github.com/AliRKhojasteh/Bubble_segmentation.git", "_temp_repo"], 
                  capture_output=True)
    subprocess.run(["git", "-C", "_temp_repo", "sparse-checkout", "set", "Demo"], capture_output=True)
    if os.path.exists("_temp_repo/Demo"):
        import shutil
        shutil.copytree("_temp_repo/Demo", DEMO_DIR)
        shutil.rmtree("_temp_repo")
        print(f"Demo images ready in '{DEMO_DIR}/'")

print("Setup complete.")

## 2. Import Libraries and Configure Device

In [ ]:
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
import pickle
import glob

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

# --- Select computation device ---
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

## 3. Configure Model and Mask Generator

Select the SAM2 model variant below. The `large` model gives the best results but requires more GPU memory (~6 GB). Use `tiny` or `small` for faster processing or limited hardware.

The mask generator parameters are tuned for bubble detection in experimental images. You can adjust them based on your image characteristics:
- `points_per_side`: Grid density for point prompts (higher = more thorough but slower)
- `pred_iou_thresh`: Minimum predicted IoU to accept a mask (higher = stricter)
- `min_mask_region_area`: Minimum bubble size in pixels to keep

In [ ]:
# ============================================================
# CONFIGURATION - Adjust these settings for your experiment
# ============================================================

# Model selection: "tiny", "small", "base_plus", "large"
MODEL_NAME = "large"

# Mask generator parameters (tuned for bubble detection)
MASK_GENERATOR_PARAMS = dict(
    points_per_side=64,           # Grid density (32=fast, 64=balanced, 128=thorough)
    points_per_batch=128,         # GPU batch size for point processing
    pred_iou_thresh=0.75,         # Predicted IoU threshold [0-1]
    stability_score_thresh=0.9,   # Stability score threshold [0-1]
    stability_score_offset=0.7,   # Stability score offset
    crop_n_layers=0,              # Multi-scale crop layers (0=off, 1=on for small bubbles)
    box_nms_thresh=0.8,           # NMS threshold for removing duplicates
    crop_n_points_downscale_factor=2,
    min_mask_region_area=10.0,    # Minimum mask area in pixels
    use_m2m=True,                 # Mask-to-mask refinement
)

# ============================================================

# Model config paths
MODEL_CONFIGS = {
    "tiny":      "configs/sam2.1/sam2.1_hiera_t.yaml",
    "small":     "configs/sam2.1/sam2.1_hiera_s.yaml",
    "base_plus": "configs/sam2.1/sam2.1_hiera_b+.yaml",
    "large":     "configs/sam2.1/sam2.1_hiera_l.yaml",
}
CHECKPOINT_NAMES = {
    "tiny": "sam2.1_hiera_tiny.pt",
    "small": "sam2.1_hiera_small.pt",
    "base_plus": "sam2.1_hiera_base_plus.pt",
    "large": "sam2.1_hiera_large.pt",
}

model_cfg = MODEL_CONFIGS[MODEL_NAME]
checkpoint_path = os.path.join(CHECKPOINT_DIR, CHECKPOINT_NAMES[MODEL_NAME])

print(f"Loading model: {MODEL_NAME}...")
sam2_model = build_sam2(model_cfg, checkpoint_path, device=device)
mask_generator = SAM2AutomaticMaskGenerator(model=sam2_model, **MASK_GENERATOR_PARAMS)
print(f"Model '{MODEL_NAME}' loaded successfully.")

## 4. Visualization Helper

In [ ]:
def show_masks(image, masks, figsize=(12, 8)):
    """Display an image with all detected masks overlaid.
    
    Args:
        image: RGB image as numpy array.
        masks: List of mask dictionaries from SAM2.
        figsize: Figure size tuple.
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Original image
    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")
    
    # Image with masks
    axes[1].imshow(image)
    if masks:
        sorted_masks = sorted(masks, key=lambda x: x["area"], reverse=True)
        overlay = np.zeros((*image.shape[:2], 4))
        for mask in sorted_masks:
            color = np.concatenate([np.random.random(3), [0.5]])
            overlay[mask["segmentation"]] = color
        axes[1].imshow(overlay)
    axes[1].set_title(f"Detected Masks: {len(masks)}")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()

## 5. Run on a Single Demo Image
Test the mask generator on one of the provided demo images before batch processing.

In [ ]:
# --- Load a demo image ---
demo_images = sorted(glob.glob(os.path.join(DEMO_DIR, "B*.png")))
if not demo_images:
    print(f"No demo images found in '{DEMO_DIR}/'. Place your images there or set DEMO_DIR.")
else:
    print(f"Found {len(demo_images)} demo images.")
    
    # Process the first image as a test
    test_image_path = demo_images[0]
    print(f"\nProcessing: {test_image_path}")
    
    image = np.array(Image.open(test_image_path).convert("RGB"))
    masks = mask_generator.generate(image)
    
    print(f"Detected {len(masks)} masks")
    show_masks(image, masks)

## 6. Batch Process Images

Process all images in a directory. Each image `BXXXX.png` produces a corresponding `BXXXX_masks.pickle` file containing the detected masks.

**For your own data:** Change `IMAGE_DIRECTORY` to point to your image folder.

In [ ]:
# ============================================================
# CONFIGURATION - Set your image directory and frame range
# ============================================================

IMAGE_DIRECTORY = DEMO_DIR                # Use demo images (or set your own path)
FILE_PATTERN = "B*.png"                   # Glob pattern to match image files

# ============================================================

# Find all matching images
image_paths = sorted(glob.glob(os.path.join(IMAGE_DIRECTORY, FILE_PATTERN)))
print(f"Found {len(image_paths)} images to process.\n")

for image_path in image_paths:
    basename = os.path.splitext(os.path.basename(image_path))[0]
    pickle_path = os.path.join(IMAGE_DIRECTORY, f"{basename}_masks.pickle")
    
    # Skip if already processed
    if os.path.exists(pickle_path):
        print(f"[skip] {basename} (already processed)")
        continue
    
    image = np.array(Image.open(image_path).convert("RGB"))
    masks = mask_generator.generate(image)
    
    with open(pickle_path, "wb") as f:
        pickle.dump(masks, f)
    
    print(f"[done] {basename}: {len(masks)} masks")

print(f"\nBatch processing complete. Results saved in '{IMAGE_DIRECTORY}/'")

## Next Steps

- **Review masks interactively:** Open `interactive_mask_editor.ipynb` to add or remove masks manually (local Jupyter only, not available on Colab)
- **Post-processing:** Open `mask_postprocessing.ipynb` to create overlay visualizations and export binary masks to MATLAB `.mat` files